# 基于 MindSpore 的 TorchDrug 论文简化复现

论文 **TorchDrug: A Powerful and Flexible Machine Learning Platform for Drug Discovery**

## 1. 检查项目路径

把项目根目录加入 `sys.path`，

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)

Project root: /home/ma-user/work/TorchDrug
Python: 3.9.11 (main, Mar 29 2022, 19:08:29) 
[GCC 7.5.0]


## 2. 导入依赖和本项目代码

这一部分主要导入 MindSpore、NumPy 和自己写的几个模块。`dataset.py` 负责数据读取和分子图构造，`models.py` 里是 GIN/GAT，`trainer.py` 里放训练和评估循环。

In [2]:
import mindspore as ms
import numpy as np

from src.dataset import load_dataset, split_dataset
from src.models import build_model
from src.trainer import TrainConfig, fit

DEVICE_TARGET = "GPU"   # 华为云 CUDA 镜像用 GPU；如果只用 CPU，把这里改成 "CPU"

ms.set_context(mode=ms.PYNATIVE_MODE, device_target=DEVICE_TARGET)
ms.set_seed(0)
np.random.seed(0)

## 3. 读取数据并生成分子图

以 BACE 为例。代码会下载公开 CSV，然后用 RDKit 把 SMILES 转成分子图。节点特征和边特征参考了 TorchDrug 的默认设置。

In [3]:
dataset_name = "bace"
dataset = load_dataset(dataset_name, PROJECT_ROOT / "data")

print("Dataset:", dataset.name)
print("Num molecules:", len(dataset))
print("Node feature dim:", dataset.node_feature_dim)
print("Edge feature dim:", dataset.edge_feature_dim)
print("First SMILES:", dataset[0]["smiles"])
print("First graph nodes:", dataset[0]["node_feature"].shape[0])

Dataset: bace
Num molecules: 1513
Node feature dim: 67
Edge feature dim: 20
First SMILES: O1CC[C@@H](NC(=O)[C@@H](Cc2cc3cc(ccc3nc2N)-c2ccccc2C)C)CC1(C)C
First graph nodes: 32


## 4. 划分训练集、验证集和测试集

BACE 这里使用 scaffold split，比例是 8:1:1。这样测试集和训练集的分子骨架差异更大，比普通随机划分更能看出模型对新结构的泛化情况。

In [4]:
train_set, valid_set, test_set = split_dataset(dataset, split="scaffold", seed=0)
print("Train / Valid / Test:", len(train_set), len(valid_set), len(test_set))

Train / Valid / Test: 1210 151 152


## 5. 构建 GIN 模型

GIN 的核心是对邻居信息做求和聚合，再用 MLP 更新节点表示。这里的 `torchdrug_like` 版本加入了边特征、BatchNorm、shortcut、concat hidden 和 sum readout，让结构尽量贴近 TorchDrug 中的写法。

In [5]:
gin_model = build_model(
    "gin",
    input_dim=dataset.node_feature_dim,
    edge_input_dim=dataset.edge_feature_dim,
    hidden_dim=128,
    num_layer=4,
    num_head=4,
    dropout=0.1,
    variant="torchdrug_like",
)
print(gin_model)

GraphClassifier<
  (layers): CellList<
    (0): TorchDrugGINLayer<
      (mlp): MLP<
        (layers): CellList<
          (0): Dense<input_channels=67, output_channels=128, has_bias=True>
          (1): Dense<input_channels=128, output_channels=128, has_bias=True>
          >
        (activation): ReLU<>
        >
      (edge_linear): Dense<input_channels=20, output_channels=67, has_bias=True>
      (batch_norm): BatchNorm1d<num_features=128, eps=1e-05, momentum=0.9, gamma=Parameter (name=layers.0.batch_norm.gamma, shape=(128,), dtype=Float32, requires_grad=True), beta=Parameter (name=layers.0.batch_norm.beta, shape=(128,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=layers.0.batch_norm.moving_mean, shape=(128,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=layers.0.batch_norm.moving_variance, shape=(128,), dtype=Float32, requires_grad=False)>
      (activation): ReLU<>
      (segment): SegmentOps<>
      >
    (1): TorchDrugGINLayer<
      

## 6. 训练并评估 GIN

为了让前面的流程检查快一点，这里只训练 3 个 epoch。

In [6]:
config = TrainConfig(epoch=3, batch_size=32, lr=1e-3, seed=0)
valid_metrics, test_metrics, selected_epoch = fit(gin_model, train_set, valid_set, test_set, config)

print("Selected epoch:", selected_epoch)
print("Best valid:", valid_metrics)
print("Matched test:", test_metrics)

epoch 001 | loss 1.4401 | valid AUROC 0.6879 AUPRC 0.9418 | test AUROC 0.7560 AUPRC 0.7189
epoch 002 | loss 0.8603 | valid AUROC 0.5143 AUPRC 0.8917 | test AUROC 0.5086 AUPRC 0.5475
epoch 003 | loss 0.7914 | valid AUROC 0.5524 AUPRC 0.8949 | test AUROC 0.5919 AUPRC 0.6323
selected best valid AUROC epoch 001
Selected epoch: 1
Best valid: {'auroc': 0.6879120879120879, 'auprc': 0.9417737213561018}
Matched test: {'auroc': 0.7560424274039298, 'auprc': 0.7188980469214967}


## 7. 构建并训练 GAT 模型

GAT 会给不同邻居分配不同注意力权重。这里同样保留边特征输入，把化学键信息加到 attention 计算里。

In [7]:
gat_model = build_model(
    "gat",
    input_dim=dataset.node_feature_dim,
    edge_input_dim=dataset.edge_feature_dim,
    hidden_dim=128,
    num_layer=4,
    num_head=4,
    dropout=0.1,
    variant="torchdrug_like",
)

config = TrainConfig(epoch=3, batch_size=32, lr=1e-3, seed=0)
valid_metrics, test_metrics, selected_epoch = fit(gat_model, train_set, valid_set, test_set, config)

print("Selected epoch:", selected_epoch)
print("Best valid:", valid_metrics)
print("Matched test:", test_metrics)

epoch 001 | loss 1.4874 | valid AUROC 0.4267 AUPRC 0.8519 | test AUROC 0.3579 AUPRC 0.4411
epoch 002 | loss 0.8498 | valid AUROC 0.5304 AUPRC 0.8868 | test AUROC 0.5123 AUPRC 0.5760
epoch 003 | loss 0.8309 | valid AUROC 0.5432 AUPRC 0.8747 | test AUROC 0.5171 AUPRC 0.5742
selected best valid AUROC epoch 003
Selected epoch: 3
Best valid: {'auroc': 0.5432234432234432, 'auprc': 0.8747160124752962}
Matched test: {'auroc': 0.5171274560945923, 'auprc': 0.5741599881833812}


## 8. 在 Notebook 中跑四组完整实验

运行所有 BACE/HIV 和 GIN/GAT 的四组实验。

In [8]:
from datetime import datetime
import random
import pandas as pd

FULL_EPOCH = 100
SEED = 0
LR = 1e-3
HIDDEN_DIM = 256
NUM_LAYER = 4
NUM_HEAD = 4
DROPOUT = 0.1
RESULT_PATH = PROJECT_ROOT / "results" / "notebook_experiment_results.csv"

experiment_plan = [
    {"dataset": "bace", "model": "gin", "split": "scaffold", "batch_size": 256},
    {"dataset": "bace", "model": "gat", "split": "scaffold", "batch_size": 256},
    {"dataset": "hiv", "model": "gin", "split": "random", "batch_size": 512},
    {"dataset": "hiv", "model": "gat", "split": "random", "batch_size": 512},
]

def reset_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    ms.set_seed(seed)

def run_one_notebook_experiment(item):
    reset_seed(SEED)
    ms.set_context(mode=ms.PYNATIVE_MODE, device_target=DEVICE_TARGET)

    dataset = load_dataset(item["dataset"], PROJECT_ROOT / "data", feature_set="torchdrug_default")
    train_set, valid_set, test_set = split_dataset(dataset, split=item["split"], seed=SEED)

    model = build_model(
        item["model"],
        input_dim=dataset.node_feature_dim,
        edge_input_dim=dataset.edge_feature_dim,
        hidden_dim=HIDDEN_DIM,
        num_layer=NUM_LAYER,
        num_head=NUM_HEAD,
        dropout=DROPOUT,
        variant="torchdrug_like",
        readout="sum",
        num_mlp_layer=1,
    )
    config = TrainConfig(
        epoch=FULL_EPOCH,
        batch_size=item["batch_size"],
        lr=LR,
        seed=SEED,
        selection="best_valid_auroc",
    )
    valid_metrics, test_metrics, selected_epoch = fit(model, train_set, valid_set, test_set, config)

    return {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "framework": "MindSpore",
        "dataset": item["dataset"],
        "model": item["model"],
        "variant": "torchdrug_like",
        "graph_format": "edge_list",
        "feature_set": "torchdrug_default",
        "seed": SEED,
        "split": item["split"],
        "selection": "best_valid_auroc",
        "selected_epoch": selected_epoch,
        "epoch": FULL_EPOCH,
        "batch_size": item["batch_size"],
        "hidden_dim": HIDDEN_DIM,
        "num_layer": NUM_LAYER,
        "num_head": NUM_HEAD,
        "readout": "sum",
        "num_mlp_layer": 1,
        "node_feature_dim": dataset.node_feature_dim,
        "edge_feature_dim": dataset.edge_feature_dim,
        "valid_auroc": valid_metrics["auroc"],
        "valid_auprc": valid_metrics["auprc"],
        "test_auroc": test_metrics["auroc"],
        "test_auprc": test_metrics["auprc"],
    }

notebook_results = []
for item in experiment_plan:
    print(f"\n=== dataset={item['dataset']} model={item['model']} split={item['split']} ===")
    notebook_results.append(run_one_notebook_experiment(item))

result_df = pd.DataFrame(notebook_results)
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
result_df.to_csv(RESULT_PATH, index=False)
print("Saved results to:", RESULT_PATH)
result_df



=== dataset=bace model=gin split=scaffold ===
epoch 001 | loss 7.3627 | valid AUROC 0.3890 AUPRC 0.8566 | test AUROC 0.3165 AUPRC 0.4256
epoch 002 | loss 2.0709 | valid AUROC 0.3418 AUPRC 0.8311 | test AUROC 0.2551 AUPRC 0.3925
epoch 003 | loss 0.9896 | valid AUROC 0.3179 AUPRC 0.8186 | test AUROC 0.2372 AUPRC 0.3866
epoch 004 | loss 0.6255 | valid AUROC 0.3220 AUPRC 0.8198 | test AUROC 0.2384 AUPRC 0.3869
epoch 005 | loss 0.5625 | valid AUROC 0.3220 AUPRC 0.8180 | test AUROC 0.2422 AUPRC 0.3883
epoch 006 | loss 0.5713 | valid AUROC 0.3158 AUPRC 0.8142 | test AUROC 0.2424 AUPRC 0.3883
epoch 007 | loss 0.5431 | valid AUROC 0.3125 AUPRC 0.8091 | test AUROC 0.2476 AUPRC 0.3905
epoch 008 | loss 0.5137 | valid AUROC 0.3092 AUPRC 0.8035 | test AUROC 0.2612 AUPRC 0.3957
epoch 009 | loss 0.4823 | valid AUROC 0.3194 AUPRC 0.8076 | test AUROC 0.2787 AUPRC 0.4032
epoch 010 | loss 0.5335 | valid AUROC 0.3359 AUPRC 0.8140 | test AUROC 0.3013 AUPRC 0.4118
epoch 011 | loss 0.5499 | valid AUROC 0.325

,timestamp,framework,dataset,model,variant,graph_format,feature_set,seed,split,selection,...,num_layer,num_head,readout,num_mlp_layer,node_feature_dim,edge_feature_dim,valid_auroc,valid_auprc,test_auroc,test_auprc
0,2026-06-17T14:18:18,MindSpore,bace,gin,torchdrug_like,edge_list,torchdrug_default,0,scaffold,best_valid_auroc,...,4,4,sum,1,67,20,0.732234,0.949506,0.685272,0.683166
1,2026-06-17T14:21:15,MindSpore,bace,gat,torchdrug_like,edge_list,torchdrug_default,0,scaffold,best_valid_auroc,...,4,4,sum,1,67,20,0.694139,0.937069,0.772561,0.758483
2,2026-06-17T14:48:44,MindSpore,hiv,gin,torchdrug_like,edge_list,torchdrug_default,0,random,best_valid_auroc,...,4,4,sum,1,67,18,0.872292,0.528496,0.785792,0.414923
3,2026-06-17T15:47:21,MindSpore,hiv,gat,torchdrug_like,edge_list,torchdrug_default,0,random,best_valid_auroc,...,4,4,sum,1,67,18,0.858467,0.499831,0.764910,0.359794


## 9. 结果记录

运行得到的结果保存到 `results/notebook_experiment_results.csv`。